<a href="https://colab.research.google.com/github/jgehbauer/spotfully/blob/master/AgenticLoop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 11.4 MB/s eta 0:00:00


In [2]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install -q google-search-results

  Preparing metadata (setup.py) ... done


In [4]:
import os
from google.colab import userdata

os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_community.utilities import SerpAPIWrapper

# Configure SerpApi for Google Scholar
serp_scholar_search = SerpAPIWrapper(params={"engine": "google_scholar"}, serpapi_api_key=os.environ["SERPAPI_API_KEY"])

def scholar_search(query: str) -> str:
    """Performs a Google Scholar search and returns the results."""
    return serp_scholar_search.run(query)

/tmp/ipykernel_4375/1980107368.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SerpAPIWrapper


In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class State(TypedDict):
    task: str
    route: str
    research: str
    analysis: str
    final: str

# Define the agents

In [6]:
def router(state: State):
    prompt = f"""
    Decide who should handle this task first:
    - researcher: if information gathering is needed
    - analyst: if the task is mainly reasoning or structuring

    Task: {state['task']}

    Answer only: researcher or analyst.
    """
    route = llm.invoke(prompt).content.strip().lower()
    return {"route": route}

In [7]:
def researcher(state: State):
    # The LLM now needs to decide what to search
    tool_decision_prompt = f"""
    Generate a concise search query for a Google Scholar search based on the following task.
    Format your response as '[your search query]'.

    Task: {state['task']}

    Search Query:
    """
    search_query = llm.invoke(tool_decision_prompt).content.strip()
    print(f"LLM generated search query: {search_query}")

    tool_name = 'Google Scholar Search'
    search_results = scholar_search(search_query)

    print(f"Using {tool_name} with query: {search_query}")
    print(f"Search results: {search_results}")

    # Finally, let the LLM synthesize the research based on search results and the original task
    messages = [
        SystemMessage(content="You are a research agent. Collect relevant facts and background based on the task and provided search results. Summarize the key findings."),
        HumanMessage(content=f"Task: {state['task']}\n\nSearch Results from {tool_name}:\n{search_results}\n\nSynthesize the key information:")
    ]
    result = llm.invoke(messages).content
    return {"research": result}

In [8]:
def analyst(state: State):
    input_text = f"""
    Task: {state['task']}

    Research notes:
    {state.get('research', '')}

    Analyze the task and structure the answer.
    """
    messages = [
        SystemMessage(content="You are an analysis agent. Create a clear structured response."),
        HumanMessage(content=input_text)
    ]
    result = llm.invoke(messages).content
    return {"analysis": result}

In [9]:
def writer(state: State):
    input_text = f"""
    Task: {state['task']}

    Research:
    {state.get('research', '')}

    Analysis:
    {state.get('analysis', '')}

    Write the final answer clearly and concisely.
    """
    result = llm.invoke(input_text).content
    return {"final": result}

# Define routing logic

In [10]:
def route_first_step(state: State) -> Literal["researcher", "analyst"]:
    return state["route"]

# Build the LangGraph

In [11]:
graph = StateGraph(State)

graph.add_node("router", router)
graph.add_node("researcher", researcher)
graph.add_node("analyst", analyst)
graph.add_node("writer", writer)

graph.add_edge(START, "router")

graph.add_conditional_edges(
    "router",
    route_first_step,
    {
        "researcher": "researcher",
        "analyst": "analyst"
    }
)

graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", END)

app = graph.compile()

# Run it

In [12]:
result = app.invoke({
    "task": "Define agentic AI",
    "route": "",
    "research": "",
    "analysis": "",
    "final": ""
})

print(result["final"])

LLM generated search query: [define agentic AI]
Using Google Scholar Search with query: [define agentic AI]
Search results: ['… The provided definition of Agentic AI allows the … Agentic AI, particularly in sectors with a scope for AI automation. Though built on basic AI principles, Agentic AI increases the scope of AI …', '… this white paper, we suggest a definition of agentic AI systems and the parties in the agentic … In Section 2, we define agentic AI systems and the human parties in the agentic AI life-cycle. …', '… of agentic AI systems by bringing together their definitions, frameworks, and architectures, and by comparing them with related areas like generative AI, … We define agentic AI’s core …', '… and Agentic AI using … with defining both GenAI and Agentic AI before elaborating on their key characteristics, discussing the need for Agentic AI and the evolution of GenAI to Agentic AI …', '… ) represents a transformative leap in AI, evolving beyond reactive systems to autonomou